In [32]:
from tools.utils import *
from time import sleep
from tqdm import tqdm
from ollama import Client
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch
import spacy
import json

In [33]:


with open('api_key.json') as f:
    api_key = json.load(f)['api_key']

client = Client(
            host="https://ollama.com",
            headers={'Authorization': api_key}
        )

def prompt_ollama_turbo(client,messages, model):
    try:
        return client.chat(model, messages=messages)
    except Exception as e:
        try:
            sleep(.1)
            return client.chat(model, messages=messages)
        except Exception as e:
            print(f"Error creating client: {e}")
            sleep(.1)
            return None
    


In [34]:
import anthropic

with open('api_key.json') as f:
    anthropic_api_key = json.load(f)['anthropic_api_key']

anthropic_client = anthropic.Anthropic(api_key=anthropic_api_key)

def prompt_claude(client, messages, model="claude-opus-5", max_tokens=1024):
    try:
        return client.messages.create(model=model, max_tokens=max_tokens, messages=messages)
    except Exception as e:
        try:
            sleep(.1)
            return client.messages.create(model=model, max_tokens=max_tokens, messages=messages)
        except Exception as e:
            print(f"Error creating client: {e}")
            sleep(.1)
            return None


In [35]:
definitions = {
    'class waiver' : """A class action waiver is a provision found in some contracts which prohibits a party from filing a class action legal proceeding against the other party, or both parties waiving the right to file class actions against each other. Most class action waiver clauses include this wording or a variation of it: You and we agree that any dispute filed against each other must be on an individual basis and not as a class or collective action.""",
    'opt-out' : """a clause that permits signatories to a contract to opt out of particular provisions, or to terminate the contract early""",
    'arbitration': """In contract law, an arbitration clause is a clause in a contract that requires the parties to resolve their disputes through an arbitration process. Although such a clause may or may not specify that arbitration occur within a specific jurisdiction, it always binds the parties to a type of resolution outside the courts, and is therefore considered a kind of forum selection clause.""",
    'modification': """This clause gives the platform rights to unilaterally change the contract at any time and states how an agreement can be changed or modified.""",
    'anti-scraping': """An anti-scraping clause is a provision in a contract that prohibits the use of automated tools or software to extract data from a website or online service without permission from the website owner. This clause is often included in the terms of service or user agreements of websites to protect their content and data from being harvested by third parties.""",
}

# Prepare data for annotation

In [5]:
# # prepare the spacy model and pipeline
# nlp = spacy.load("en_core_web_sm")
# nlp.disable_pipes("tagger", "parser", "attribute_ruler", "lemmatizer")
# nlp.add_pipe('sentencizer')


In [13]:
processed_data_dir = Path('processed_data/tous')

In [14]:
embeddings = np.loadtxt(processed_data_dir / 'embedding_bge.tsv')
metadata = pd.read_csv(processed_data_dir / 'metadata.tsv',sep='\t')
print(len(embeddings), len(metadata))

124495 124495


# Clause classification

In [75]:
clause_type = 'anti-scraping' #'anti-scraping' # 'modification' |'opt-out'  |'arbitration' | 'opt-out' | 'class waiver' | 'anti-scraping'
examples_df = pd.read_excel(f'annotations/seed_examples/{clause_type}_clauses.xlsx', sheet_name='Sheet1')

In [76]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
embedding_model_name = "bge-large-en-v1.5"
model, prefix = load_embedding_model(embedding_model_name, device=device) # load the embedding model and its task-instruction text prefix, and send it to the device


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [77]:
examples_df['processed_text'] = examples_df['Examples'].apply(lambda x: prefix + x.lower().strip().replace('\n', ' '))

In [78]:
target_embeddings = model.encode(examples_df['processed_text'].tolist())

In [79]:
average_embedding = target_embeddings.mean(axis=0)

In [80]:

metadata['tou_id'] = metadata.apply(lambda x: f"{x['platform']}_{x['year']}", axis=1)

In [81]:
#dist_sample_max

In [82]:
def get_surrounding_sentences(indices, metadata, id_col='tou_id', text_col='sentence', window=5):
    """For each row position in `indices`, collect up to `window` sentences
    before and after it from `metadata`, restricted to rows sharing the same
    `id_col` value (so context never crosses into a different document).

    Returns a DataFrame with columns: id, tou_id, Target_sentence,
    Previous_sents, Next_sents (the latter two are lists, possibly shorter
    than `window` near a document boundary or the edge of `metadata`).
    """
    rows = []
    for idx in indices:
        idx = int(idx)
        tou_id = metadata.loc[idx, id_col]
        target_sentence = metadata.loc[idx, text_col]

        prev_candidates = metadata.iloc[max(idx - window, 0):idx]
        previous_sents = prev_candidates.loc[prev_candidates[id_col] == tou_id, text_col].tolist()

        next_candidates = metadata.iloc[idx + 1:idx + 1 + window]
        next_sents = next_candidates.loc[next_candidates[id_col] == tou_id, text_col].tolist()

        rows.append({
            'id': idx,
            'tou_id': tou_id,
            'Target_sentence': target_sentence,
            'Previous_sents': previous_sents,
            'Next_sents': next_sents,
        })

    return pd.DataFrame(rows)




In [83]:
similarity_scores_max = cosine_similarity(embeddings, target_embeddings)
similarity_scores_max = similarity_scores_max.max(axis=1)
top_n_position_max = np.argpartition(similarity_scores_max, -500)[-500:][::-1]
np.random.seed(42)
dist_sample_max = np.random.choice(list(range(len(metadata))), p=(similarity_scores_max / similarity_scores_max.sum()), size=500, replace=False)

In [84]:
similarity_scores_av = cosine_similarity(embeddings, [average_embedding]).flatten()
top_n_position_av = np.argpartition(similarity_scores_av, -500)[-500:][::-1]
np.random.seed(42)
dist_sample_av = np.random.choice(list(range(len(metadata))), p=(similarity_scores_av / similarity_scores_av.sum()), size=500, replace=False)


In [85]:
context_df_max = get_surrounding_sentences(np.concatenate([top_n_position_max, dist_sample_max]), metadata, window=5)
context_df_av = get_surrounding_sentences(np.concatenate([top_n_position_av, dist_sample_av]), metadata, window=5)
context_df = pd.concat([context_df_max, context_df_av], ignore_index=True)
context_df.drop_duplicates(subset=['Target_sentence'], inplace=True)
#context_df['labels'] = 0
context_df.to_csv(f'annotations/similar_examples/{clause_type}_selected_sentences.csv')

In [86]:

import random
threshold = 2000
numbers = list(range(1,5)) + [i*-1 for i in list(range(1,5))]
top_n_position_max = np.argpartition(similarity_scores_max, -threshold)[-threshold:][::-1]
top_n_position_max_context = top_n_position_max + np.array(random.choices(numbers, k=top_n_position_max.shape[0]))
display(top_n_position_max[:5], top_n_position_max_context[:5])
context_df_max_context = get_surrounding_sentences(top_n_position_max_context, metadata, window=5)
context_df_max_context.drop_duplicates(subset=['Target_sentence'], inplace=True)
context_df_max_context.to_csv(f'annotations/similar_examples/{clause_type}_selected_sentences_context.csv', index=False)


array([ 31818,  18916, 117550,  28839, 117549])

array([ 31822,  18919, 117551,  28838, 117551])

In [56]:
context_df_max_context.shape, len(top_n_position_max_context)

((665, 5), 1000)

In [57]:
# sentences_positions = list(set(np.concatenate([top_n_position_max, top_n_position_av, dist_sample_max, dist_sample_av])))
# print(f'Total sentences selected: {len(sentences_positions)}')

In [58]:


# out_csv = pd.DataFrame(metadata.iloc[sentences_positions].sentence.unique())
# out_csv.columns = ['sentence']
# out_csv['label'] = 0
# #out_csv['similarity'] = similarity_scores[top_n_position]
# out_csv.to_csv(f'annotations/similar_examples/{clause_type}_selected_sentences.csv')

## Pre-annotate selected examples with Claude

In [59]:
# prepare examples for annotation
df = pd.read_csv(f'annotations/similar_examples/{clause_type}_selected_sentences_context.csv', index_col=0)
df.columns,df.shape


(Index(['tou_id', 'Target_sentence', 'Previous_sents', 'Next_sents'], dtype='object'),
 (665, 4))

In [60]:
df.head(5)

,tou_id,Target_sentence,Previous_sents,Next_sents
id,,,,
116201,instagram_20130101,There may be times when we offer a special fea...,"['To access our previous Terms of Use, please ...","['In those cases, the terms specific to the sp..."
11937,pinterest_20120401,You may not use the Service if you are a resid...,['THE LIMITATIONS OF LIABILITY SET FORTH ABOVE...,"['Unless otherwise explicitly stated, all mate..."
105321,meetup_20080901,"Unless otherwise agreed by the parties, the me...","[""The parties will cooperate with JAMS and wit...","['13.4 Arbitration.', 'The parties agree that ..."
84831,threads_20231201,"For clarity, the Protocol is not part of the T...","['By using the Threads Service, you agree to t...",['Third Party Content means content and inform...
92737,bluesky_20241201,"If we need to go to court, it will be in Delaw...",['YOU ACKNOWLEDGE THAT BLUESKY SHALL NOT BE RE...,['You agree not to participate in a class-acti...


In [61]:
df['prompt'] = df.apply(lambda s: """
You are a helpful AI that classifies a given sentence as indicating the presence of a {0} clause (or not). 

We provide you with the following information in this order
- DEFINITION: a definition of a(n) {0} clause followed
- EXAMPLES: three examples of {0} clauses. 
- CONTEXT: The surrounding context of the sentence, the previous sentences and the next sentences, which may help you determine whether the sentence contains a {0} clause. The location of the target sentence is indicated by the text '<TARGET>' in the context.
- SENTENCE: the sentence to classify
          
DEFINITION: 
The definition of a(n) {0} clause is: {3}

EXAMPLES:          
Below are three examples of {0} clauses.\n\n{1}

CONTEXT:
{4}

SENTENCE:
{2}

Does SENTENCE indicate the presence of a {0} clause? It does not need to be the complete clause, it can be part of the clause or contain an indication that the text contains a {0} clause.
Please reply only 'yes' or 'no', nothing else.
""".format(
     clause_type,examples_df['Examples'].sample(3).str.cat(sep='\n'),
               s['Target_sentence'], 
               definitions[clause_type],
               '\n'.join(eval(s['Previous_sents'])) +  ' <TARGET> '  + '\n'.join(eval(s['Next_sents'])))
               , axis=1)




In [62]:
print(df['prompt'].iloc[5])


You are a helpful AI that classifies a given sentence as indicating the presence of a arbitration clause (or not). 

We provide you with the following information in this order
- DEFINITION: a definition of a(n) arbitration clause followed
- EXAMPLES: three examples of arbitration clauses. 
- CONTEXT: The surrounding context of the sentence, the previous sentences and the next sentences, which may help you determine whether the sentence contains a arbitration clause. The location of the target sentence is indicated by the text '<TARGET>' in the context.
- SENTENCE: the sentence to classify
          
DEFINITION: 
The definition of a(n) arbitration clause is: In contract law, an arbitration clause is a clause in a contract that requires the parties to resolve their disputes through an arbitration process. Although such a clause may or may not specify that arbitration occur within a specific jurisdiction, it always binds the parties to a type of resolution outside the courts, and is the

In [63]:
#model_list = ["mistral-large-3:675b-cloud", "gpt-oss:120b", "deepseek-v3.1:671b-cloud"]
model_list = ['claude-haiku-4-5-20251001']

In [64]:
for model in model_list:
    df[f'response_{model}'] = None

In [65]:


# message = anthropic_client.messages.create(
#     model="claude-opus-5",
#     max_tokens=1000,
#     messages=[
#         {
#             "role": "user",
#             "content": "What should I search for to find the latest developments in renewable energy?",
#         }
#     ],
# )

# for block in message.content:
#     if block.type == "text":
#         print(block.text)

In [66]:
tqdm.pandas()
for model in model_list:
    df[f'response_{model}'] = df.progress_apply(
            lambda row: prompt_claude(anthropic_client,[{"role": "user", "content": row['prompt']}], model=model) if pd.isna(row[f'response_{model}']) else row[f'response_{model}'], axis=1
        )   

  0%|          | 0/665 [00:00<?, ?it/s]

100%|██████████| 665/665 [09:12<00:00,  1.20it/s]


In [67]:
df[f'response_{model}'].iloc[0].content[0].text

'No'

In [68]:
df[f'response_{model}_parsed'] =df[f'response_{model}'].apply(lambda x: x.content[0].text.lower() if x is not None else None)#.values_counts()#.to_csv(f'annotations/ollama_annotations/{clause_type}_responses_{model}.csv')
df[f'response_{model}_parsed'].value_counts()

response_claude-haiku-4-5-20251001_parsed
no     411
yes    254
Name: count, dtype: int64

In [69]:
for model in model_list:
    df[f'label_{model}'] = df[f'response_{model}_parsed'].apply(lambda x: 1 if x in ['yes', 'yes.'] else 0 if x in ['no', 'no.'] else None)
    df[f'label_{model}'].value_counts()

In [70]:
df[f'label_{model}'].value_counts()

label_claude-haiku-4-5-20251001
0    411
1    254
Name: count, dtype: int64

In [71]:
df[df[f'label_{model}'] == 0]

,tou_id,Target_sentence,Previous_sents,Next_sents,prompt,response_claude-haiku-4-5-20251001,response_claude-haiku-4-5-20251001_parsed,label_claude-haiku-4-5-20251001
id,,,,,,,,
116201,instagram_20130101,There may be times when we offer a special fea...,"['To access our previous Terms of Use, please ...","['In those cases, the terms specific to the sp...",\nYou are a helpful AI that classifies a given...,"Message(id='msg_011CeEZ33ZGZeyiYbVn2HnZH', con...",no,0
11937,pinterest_20120401,You may not use the Service if you are a resid...,['THE LIMITATIONS OF LIABILITY SET FORTH ABOVE...,"['Unless otherwise explicitly stated, all mate...",\nYou are a helpful AI that classifies a given...,"Message(id='msg_011CeEZ36DG5iyYEsH8hfqgX', con...",no,0
105321,meetup_20080901,"Unless otherwise agreed by the parties, the me...","[""The parties will cooperate with JAMS and wit...","['13.4 Arbitration.', 'The parties agree that ...",\nYou are a helpful AI that classifies a given...,"Message(id='msg_011CeEZ38gLp7PAhzfs1DkhW', con...",no,0
84831,threads_20231201,"For clarity, the Protocol is not part of the T...","['By using the Threads Service, you agree to t...",['Third Party Content means content and inform...,\nYou are a helpful AI that classifies a given...,"Message(id='msg_011CeEZ3AwXAq2yvG2QWLVFQ', con...",no,0
92737,bluesky_20241201,"If we need to go to court, it will be in Delaw...",['YOU ACKNOWLEDGE THAT BLUESKY SHALL NOT BE RE...,['You agree not to participate in a class-acti...,\nYou are a helpful AI that classifies a given...,"Message(id='msg_011CeEZ3DV55NJmjALU8A5GG', con...",no,0
...,...,...,...,...,...,...,...,...
13035,match_20250901,We have included brief summaries at the beginn...,"['The Companys business is conducted, in part,...",['The summaries do not replace the text of eac...,\nYou are a helpful AI that classifies a given...,"Message(id='msg_011CeEZjKz75XK9pD2GjojjL', con...",no,0
111905,twitch_20201001,ii) If you are a resident in any jurisdiction ...,['Notice to Twitch shall be sent to: Twitch In...,['e. Claims YOU AND TWITCH AGREE THAT ANY CAU...,\nYou are a helpful AI that classifies a given...,"Message(id='msg_011CeEZjNqzoYg5WoVnRiWgK', con...",no,0
32973,grindr_20250901,The Notice must be signed by the party initiat...,"['If You are represented by counsel, Your coun...","['The Notice must include: (1) Your name, phon...",\nYou are a helpful AI that classifies a given...,"Message(id='msg_011CeEZjRLpBRfT28ZkcEX4T', con...",no,0


In [72]:
df.drop('labels', axis=1, inplace=True)

KeyError: "['labels'] not found in axis"

In [73]:
df.columns

Index(['tou_id', 'Target_sentence', 'Previous_sents', 'Next_sents', 'prompt',
       'response_claude-haiku-4-5-20251001',
       'response_claude-haiku-4-5-20251001_parsed',
       'label_claude-haiku-4-5-20251001'],
      dtype='object')

In [74]:
Path('annotations/claude_annotations').mkdir(parents=True, exist_ok=True)
df.to_csv(f'annotations/claude_annotations/{clause_type}_annotations_context.csv')

In [ ]:
# #df['sum'] = df[[c for c in df.columns if c.startswith('label_')]].sum(axis=1)
# #df[['sentence', 'sum'] + [c for c in df.columns if c.startswith('label_')]].to_csv(f'annotations/ollama_annotations/{clause_type}_labels.csv')
# df.to_csv(f'annotations/claude_annotations/{clause_type}_annotations.csv')

In [173]:
df.columns

Index(['id', 'tou_id', 'Target_sentence', 'Previous_sents', 'Next_sents',
       'prompt', 'response_claude-haiku-4-5-20251001',
       'response_claude-haiku-4-5-20251001_parsed',
       'label_claude-haiku-4-5-20251001'],
      dtype='object')

# Fin.